# v6: Follow-up handling and missingness features (no gain, reverted)

**Version:** v6  
**Kaggle submission:** none (experiment; reverted)

**Hypotheses.** (B) the missing-value count predicts failure, so explicit NaN-count features might help the trees. (C/D) healthy vehicles with short follow-up have unreliable labels (v1): dropping them (C: follow-up < 180 d) or down-weighting them (D: weight min(followup,365)/365) should give cleaner training labels. (E) B + D. Only the **classifier** changes; the v5 pipeline is held fixed and every variant is scored on **all** validation vehicles. `followup_days` shapes training rows/weights only, never a feature.

*Self-contained: needs only `../data/train.csv` and `../data/test.csv`. Part of the Fleet Maintenance project (Kaggle `mlprojectfleet2`); the cost, decision rule and validation protocol are derived in `fleet_maintenance.ipynb`.*

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xgboost as xgb
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score, brier_score_loss, log_loss
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split, cross_val_predict

DATA, OUT = Path("../data"), Path("outputs"); OUT.mkdir(exist_ok=True)
SEED = 42
INSPECT, UNDER, OVER = 10.0, 0.20, 0.02          # inspection cost; shortage cost / unit; unused-budget cost / unit
Q_STAR = UNDER / (UNDER + OVER)                  # newsvendor critical ratio 0.909
HISTB = [f"s{i:03d}" for i in range(54, 64)]     # 10-bin histogram that is shifted by one bin in test

def load(raw=False):
    """train/test with the v1 fixes: realign s054-s063 (test bin k+1 == train bin k), drop s047 (all NaN in
    test) and s146 (constant). raw=True returns the files untouched."""
    tr, te = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
    if not raw:
        te[HISTB[:-1]] = te[HISTB[1:]].to_numpy(); te[HISTB[-1]] = np.nan
        tr, te = tr.drop(columns=["s047", "s146"]), te.drop(columns=["s047", "s146"])
    return tr, te, [c for c in tr.columns if c.startswith("s")]

tr, te, S = load()
dev_idx, ho_idx = train_test_split(np.arange(len(tr)), test_size=0.2, stratify=tr.failure, random_state=SEED)
dev, ho = tr.iloc[dev_idx].reset_index(drop=True), tr.iloc[ho_idx].reset_index(drop=True)   # ho: locked 20% hold-out
y, c = dev.failure.to_numpy(), dev.repair_cost.fillna(0).to_numpy() * dev.failure.to_numpy()
y_ho, c_ho = ho.failure.to_numpy(), ho.repair_cost.fillna(0).to_numpy() * ho.failure.to_numpy()
c_all = tr.repair_cost.fillna(0).to_numpy() * tr.failure.to_numpy()
F = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))   # the 5 fixed folds
print(f"train {tr.shape}  test {te.shape}  sensors {len(S)}  dev {len(dev)} / hold-out {len(ho)}  failures {int(y.sum())}")

train (56000, 172)  test (20000, 169)  sensors 168  dev 44800 / hold-out 11200  failures 318


In [2]:
def vehicle_cost(c, flag, b):
    return np.where(flag == 1, INSPECT + UNDER * np.maximum(c - b, 0) + OVER * np.maximum(b - c, 0), c)

def per_vehicle(c, flag, b):
    return float(vehicle_cost(np.asarray(c, float), np.asarray(flag), np.asarray(b, float)).mean())

fold_cost = lambda flag, b: np.array([per_vehicle(c[v], flag[v], b[v]) for _, v in F])

def write_submission(ids, flag, budget, name):
    flag, budget = np.asarray(flag).astype(int), np.maximum(np.asarray(budget, float), 0)
    assert len(ids) == 20000 and ids.is_unique and set(flag) <= {0, 1} and np.isfinite(budget).all()
    pd.DataFrame({"id": ids, "flag": flag, "budget": budget.round(2)}).to_csv(OUT / f"submission_{name}.csv", index=False)
    print(f"wrote outputs/submission_{name}.csv: flagged {flag.mean():.2%} of test vehicles")

def decide_global(p, fail_costs):
    """Same repair-cost distribution F (empirical failure costs) for every vehicle."""
    p = np.clip(np.asarray(p, float), 0, 1); fc = np.sort(np.asarray(fail_costs, float)); mu = fc.mean()
    b = np.zeros_like(p); need = p > 1 - Q_STAR
    b[need] = np.quantile(fc, 1 - (1 - Q_STAR) / p[need])                       # newsvendor budget
    fail = (UNDER * np.maximum(fc[None, :] - b[:, None], 0) + OVER * np.maximum(b[:, None] - fc[None, :], 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b   # flag iff cheaper than not flagging

def decide_pv(p, m, resid, fail_costs=None, K=99, scale=None):
    """Per-vehicle cost distribution: log c | fail ~ m_i + scale_i * residuals (out-of-fold residuals).
    fail_costs given -> budget from that GLOBAL distribution (ablation); scale -> per-vehicle spread (v9)."""
    p, m = np.clip(np.asarray(p, float), 0, 1), np.asarray(m, float)
    sc = np.ones_like(m) if scale is None else np.asarray(scale, float)
    M = np.exp(m[:, None] + sc[:, None] * np.quantile(resid, (np.arange(K) + 0.5) / K)[None, :]); mu = M.mean(1)
    b = np.zeros_like(p); need = p > 1 - Q_STAR; level = 1 - (1 - Q_STAR) / p[need]
    b[need] = (np.exp(m[need] + sc[need] * np.quantile(resid, level)) if fail_costs is None
               else np.quantile(np.asarray(fail_costs, float), level))
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b

class Platt:
    def fit(self, p, y):
        self.m = LogisticRegression(C=1e6, max_iter=1000).fit(self._x(p), y); return self
    def predict(self, p): return self.m.predict_proba(self._x(p))[:, 1]
    @staticmethod
    def _x(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))[:, None]

class Iso:
    def fit(self, p, y):
        self.m = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y); return self
    def predict(self, p): return self.m.predict(p)

make_cal = lambda kind: {"platt": Platt, "iso": Iso}[kind]()

PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=1, n_jobs=-1, random_state=SEED)                         # failure classifier (v2..)
CM = dict(n_estimators=200, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.5,
          min_child_weight=5, n_jobs=-1, random_state=SEED)                             # repair-cost model (v5)
inner = KFold(5, shuffle=True, random_state=SEED)

def cost_model(params, Xa, lc, Xb):
    """XGBoost on log repair cost (failures only). Returns predictions for Xb and OUT-OF-FOLD residuals of Xa."""
    resid = lc - cross_val_predict(xgb.XGBRegressor(**params), Xa, lc, cv=inner)
    return xgb.XGBRegressor(**params).fit(Xa, lc).predict(Xb), resid

def fit_oof(X, params=PARAMS):
    """5-fold out-of-fold failure probabilities on dev, and probabilities for test from a model fit on all train."""
    o = np.zeros(len(dev))
    for a, b in F:
        o[b] = xgb.XGBClassifier(**params).fit(X.iloc[a], y[a]).predict_proba(X.iloc[b])[:, 1]
    return o

shift = lambda p, k: k * p / (1 - p) / (1 + k * p / (1 - p))       # prior shift by odds factor k

In [3]:
def pipeline_costs(oofs, base="A", cm=CM, X=None, repeats=None):
    """Paired CV cost for several failure-probability candidates through the v5 pipeline
    (nested cost model fitted once per fold, Platt fitted on the other folds' OOF)."""
    X = dev[S] if X is None else X
    cost = {k: [] for k in oofs}
    for a, b in F:
        fa = a[y[a] == 1]
        m_b, resid = cost_model(cm, X.iloc[fa], np.log(c[fa]), X.iloc[b])
        for k, o in oofs.items():
            fl, bu = decide_pv(make_cal("platt").fit(o[a], y[a]).predict(o[b]), m_b, resid)
            cost[k].append(per_vehicle(c[b], fl, bu))
    return {k: np.array(v) for k, v in cost.items()}

def report(cost, base):
    A = cost[base]
    for k, x in cost.items():
        d = x - A; print(f"{k:12s} CV cost {x.mean():.4f} ± {x.std():.3f}   paired vs {base}: {d.mean():+.4f}   per fold {np.round(d, 3)}   better {int((d < 0).sum())}/{len(d)}")

In [4]:
fu = dev.followup_days.to_numpy()
def with_miss(df):
    X = df[S].copy(); S2 = [s for s in S if s != "s063"]            # s063 is NaN by construction in test
    X["n_nan"] = df[S2].isna().sum(axis=1)
    for i in range(0, len(S2), 20): X[f"nan_blk{i // 20}"] = df[S2[i:i + 20]].isna().sum(axis=1)
    return X
Xb = with_miss(dev)
W = lambda a: np.where(y[a] == 1, 1.0, np.minimum(fu[a], 365) / 365)
variants = {"A": (dev[S], None, None), "B": (Xb, None, None),
            "C": (dev[S], lambda a: a[(y[a] == 1) | (fu[a] >= 180)], None),
            "D": (dev[S], None, W), "E": (Xb, None, W)}
oofs = {}
for k, (X, sel, wf) in variants.items():
    o = np.zeros(len(dev))
    for a, b in F:
        aa = sel(a) if sel else a
        o[b] = xgb.XGBClassifier(**PARAMS).fit(X.loc[aa], y[aa], sample_weight=None if wf is None else wf(aa)).predict_proba(X.loc[b])[:, 1]
    oofs[k] = o
    print(f"{k}: OOF AUC {roc_auc_score(y, o):.4f}  PR-AUC {average_precision_score(y, o):.4f}  mean p {o.mean():.4f}")
cost = pipeline_costs(oofs); print(); report(cost, "A")

A: OOF AUC 0.9840  PR-AUC 0.5263  mean p 0.0058


B: OOF AUC 0.9846  PR-AUC 0.5258  mean p 0.0058


C: OOF AUC 0.9843  PR-AUC 0.5171  mean p 0.0077


D: OOF AUC 0.9841  PR-AUC 0.5324  mean p 0.0068


E: OOF AUC 0.9841  PR-AUC 0.5177  mean p 0.0068



A            CV cost 0.9037 ± 0.235   paired vs A: +0.0000   per fold [0. 0. 0. 0. 0.]   better 0/5
B            CV cost 0.9145 ± 0.247   paired vs A: +0.0108   per fold [ 0.009 -0.003  0.015 -0.009  0.042]   better 2/5
C            CV cost 0.8878 ± 0.178   paired vs A: -0.0160   per fold [-0.011  0.06   0.027  0.019 -0.174]   better 2/5
D            CV cost 0.8808 ± 0.180   paired vs A: -0.0230   per fold [ 0.025  0.047  0.029 -0.033 -0.183]   better 2/5
E            CV cost 0.9229 ± 0.245   paired vs A: +0.0192   per fold [ 0.039  0.043  0.019 -0.031  0.025]   better 1/5


## Results and decision: revert to A

| Variant | OOF AUC | CV cost | Paired Δ vs A | Folds better |
|---|---|---|---|---|
| A (v5 Platt) | 0.9840 | 0.904 ± 0.235 | n/a | n/a |
| B: + NaN counts | 0.9846 | 0.915 | +0.011 | 2/5 |
| C: drop follow-up < 180 | 0.9843 | 0.888 ± 0.178 | −0.016 | 2/5 |
| D: weight by follow-up | 0.9841 | 0.881 ± 0.181 | −0.023 | 2/5 |
| E: B + D | 0.9841 | 0.923 | +0.019 | 1/5 |

- **No variant is a real improvement.** C and D look better on average, but that comes from **one fold** (−0.17 / −0.18 in fold 5) while they are worse in 3 of the other 4 folds: one lucky large failure, not a gain. B does not help because trees already see NaN natively.
- AUC is flat for all variants (0.984-0.985).

**Caveats:** with 397 failures, swings of 0.1-0.2 between folds are normal; a repeated-CV test would be needed to settle C/D, and the potential gain is too small to justify it. Dropping/weighting changes the training base rate (mean p 0.0058 → 0.0077 for C); Platt absorbs this, but only for the training depot. **Decision: revert; no submission.**